# 🧠 NeuroMentor APK Builder

This notebook builds a debug APK for the NeuroMentor Kivy app.

## Instructions
1. **Upload your project**: Click the folder icon (📁) in the left sidebar
2. **Run all cells**: Runtime → Run all (Ctrl+F9)
3. **Download APK**: The last cell will auto-download the APK to your computer

⏱️ **First build takes ~20-30 minutes** (downloads SDK/NDK).

## Step 1: Install system dependencies

In [ ]:
!sudo apt update -qq
!sudo apt install -y -qq build-essential git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev cmake libffi-dev libssl-dev automake ccache
print('✓ System dependencies installed')

## Step 2: Install Buildozer and Cython

In [ ]:
!pip install --upgrade buildozer cython==3.0.10
print('✓ Buildozer installed')

## Step 3: Upload project files

**Run this cell**, then use the file picker to select and upload **the zip file** of your project.

### Before running this cell:
On your PC, zip your project folder:
1. Open File Explorer
2. Navigate to `C:\Users\anish\Desktop\work\nm-git\NeuroMentor\flutter\Kivy_Section\neuro_mentor_kivy`
3. Select all files (Ctrl+A), right-click → Send to → Compressed (zipped) folder
4. Name it `neuromentor.zip`

In [ ]:
import os
from google.colab import files

# Upload the zip file
print('📁 Select your neuromentor.zip file...')
uploaded = files.upload()

# Find the uploaded zip
zip_name = list(uploaded.keys())[0]
print(f'\n✓ Uploaded: {zip_name} ({len(uploaded[zip_name]) / 1024 / 1024:.1f} MB)')

# Create project directory and extract
!mkdir -p /content/neuromentor
!unzip -o "{zip_name}" -d /content/neuromentor

# Handle case where zip contains a subfolder
items = os.listdir('/content/neuromentor')
# Filter out __MACOSX and other junk
items = [i for i in items if not i.startswith('__') and not i.startswith('.')]
if len(items) == 1 and os.path.isdir(f'/content/neuromentor/{items[0]}'):
    # Zip contained a single subfolder, move contents up
    subfolder = items[0]
    !mv /content/neuromentor/{subfolder}/* /content/neuromentor/
    !mv /content/neuromentor/{subfolder}/.* /content/neuromentor/ 2>/dev/null || true
    !rmdir /content/neuromentor/{subfolder}

os.chdir('/content/neuromentor')
print(f'\n✓ Project extracted to /content/neuromentor')
print(f'Files: {os.listdir(".")}')

# Verify main.py exists
assert os.path.exists('main.py'), '❌ main.py not found! Check your zip file.'
assert os.path.exists('buildozer.spec'), '❌ buildozer.spec not found!'
print('\n✓ main.py and buildozer.spec found')

## Step 4: Clean and exclude unnecessary files

In [ ]:
import shutil

# Remove files that should not be in the APK
for d in ['.venv', '__pycache__', '.git', 'bin', '.buildozer']:
    if os.path.isdir(d):
        shutil.rmtree(d)
        print(f'  Removed {d}/')

# Remove __pycache__ recursively
for root, dirs, files_list in os.walk('.'):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

# Remove stray temp files
for f in os.listdir('.'):
    if f.endswith(('.log', '.txt')) and f not in ['requirements.txt', 'users.json']:
        if f.startswith(('crash', 'error', 'compat')):
            os.remove(f)
            print(f'  Removed {f}')

print('\n✓ Project cleaned')

## Step 5: Build the debug APK

⏱️ **This takes 15-30 minutes on first run.** Subsequent runs take 3-5 minutes.

In [ ]:
os.chdir('/content/neuromentor')
!buildozer android debug 2>&1 | tail -50

# Check if APK was created
import glob
apks = glob.glob('bin/*.apk')
if apks:
    apk = apks[0]
    size_mb = os.path.getsize(apk) / 1024 / 1024
    print(f'\n{"="*60}')
    print(f'✅ APK BUILT SUCCESSFULLY!')
    print(f'   File: {apk}')
    print(f'   Size: {size_mb:.1f} MB')
    print(f'{"="*60}')
else:
    print('\n❌ BUILD FAILED — No APK found in bin/')
    print('Scroll up to see the error. Common fixes:')
    print('  1. Check buildozer.spec requirements')
    print('  2. Try removing scikit-learn from requirements for a simpler build')

## Step 6: Download the APK to your computer

In [ ]:
from google.colab import files
import glob

apks = glob.glob('bin/*.apk')
if apks:
    apk = apks[0]
    print(f'📲 Downloading {apk}...')
    files.download(apk)
    print('\n✓ APK downloaded! Transfer it to your phone and install.')
    print('  Make sure "Install from unknown sources" is enabled.')
else:
    print('❌ No APK found. Run the build cell (Step 5) first.')